#  Multi-class Chest X-ray Disease Classification Using Transfer Learning — 2 Stage Fine-Tuning (DenseNet121)

## Overview

Two-stage transfer-learning pipeline for 7-class chest X-ray
classification (NIH Chest X-ray14 dataset). Loads the best Stage 1
checkpoint, unfreezes the full DenseNet121 backbone, and fine-tunes
end-to-end with a low learning rate.

> Assumes the dataset is already split into `train/`, `val/`, `test/`
> folders (one sub-folder per class). Update paths in **Configuration**
> to match your environment.


## 1. Import Libraries

Data handling, modeling, evaluation, and visualization dependencies.


In [ ]:
import os
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

## 2. Define Dataset Paths & Configuration

All paths, class definitions, and hyperparameters, in one place.

> Replace the paths below with your local locations.


In [ ]:
# ---------- Raw metadata ----------
CSV_PATH = r"D:\NIH Dataset\Data_Entry_2017.csv"

# ---------- Pre-split image folders (train/val/test), one sub-folder per class ----------
DATASET_ROOT = r"D:\NIH Dataset\NIH_7class_split"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

# ---------- Model checkpoints ----------
STAGE1_MODEL_PATH = r"D:\NIH Dataset\Results\Stage1\best_model.pth"
CHECKPOINT_PATH = r"D:\NIH Dataset\checkpoint.pth"
BEST_MODEL_PATH = r"D:\NIH Dataset\best_model.pth"

# ---------- Output directory for metrics, plots, and reports ----------
RESULTS_DIR = r"D:\NIH Dataset\Results"

# ---------- Class definitions ----------
SELECTED_CLASSES = [
    "No Finding",
    "Atelectasis",
    "Effusion",
    "Cardiomegaly",
    "Nodule",
    "Mass",
    "Pneumothorax",
]
LABEL_MAP = {name: idx for idx, name in enumerate(SELECTED_CLASSES)}
NUM_CLASSES = len(SELECTED_CLASSES)

# ---------- Sampling & split configuration ----------
RANDOM_STATE = 42
NO_FINDING_SAMPLE_SIZE = 10_000  # "No Finding" is downsampled to reduce class imbalance
VAL_TEST_SPLIT = 0.2             # 20% held out for validation + test
TEST_SPLIT_OF_HELD_OUT = 0.5     # split the held-out 20% evenly into val/test

# ---------- Training hyperparameters ----------
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
NUM_WORKERS = 2

# Stage 1 achieved this validation accuracy; Stage 2 only saves a new
# "best model" checkpoint if it beats this baseline.
BEST_VAL_ACC_BASELINE = 37.48

## 3. Device Configuration

Use GPU if available, otherwise CPU.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

## 4. Load Metadata

Load `Data_Entry_2017.csv`, keep single-label images only, and filter
to the 7 target classes.


In [ ]:
df = pd.read_csv(CSV_PATH)
print("Total images in CSV:", len(df))

# Keep single-label images only (exclude multi-label rows such as "A|B")
df_single = df[~df["Finding Labels"].str.contains(r"\|")]

# Restrict to the 7 target classes
df_7class = df_single[df_single["Finding Labels"].isin(SELECTED_CLASSES)]
print("Images after filtering to 7 classes:", len(df_7class))

df_7class["Finding Labels"].value_counts()

## 5. Balance the Dataset

Downsample the dominant `No Finding` class; keep all other classes as-is.


In [ ]:
no_finding = df_7class[df_7class["Finding Labels"] == "No Finding"].sample(
    n=NO_FINDING_SAMPLE_SIZE, random_state=RANDOM_STATE
)
remaining_classes = df_7class[df_7class["Finding Labels"] != "No Finding"]

df_final = pd.concat([no_finding, remaining_classes], ignore_index=True)

print(df_final["Finding Labels"].value_counts())

## 6. Encode Labels

Map text labels to integer class indices.


In [ ]:
df_final["label"] = df_final["Finding Labels"].map(LABEL_MAP)

## 7. Train / Validation / Test Split

Stratified 80/10/10 split by class label.


In [ ]:
train_df, temp_df = train_test_split(
    df_final,
    test_size=VAL_TEST_SPLIT,
    stratify=df_final["label"],
    random_state=RANDOM_STATE,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SPLIT_OF_HELD_OUT,
    stratify=temp_df["label"],
    random_state=RANDOM_STATE,
)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}  Total: {len(df_final)}")
print("\nTrain label distribution:")
print(train_df["Finding Labels"].value_counts())
print("\nValidation label distribution:")
print(val_df["Finding Labels"].value_counts())
print("\nTest label distribution:")
print(test_df["Finding Labels"].value_counts())

## 8. Image Transforms

Training uses augmentation (flip, rotation); eval uses resize +
ImageNet normalization only.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 9. Datasets & Data Loaders

Load pre-split image folders and wrap them in `DataLoader`s.


In [ ]:
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = ImageFolder(TEST_DIR, transform=eval_transform)

print("Classes:", train_dataset.classes)
print("Class to index mapping:", train_dataset.class_to_idx)
print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

## 10. Model Architecture

DenseNet121 (ImageNet weights) with a new 7-class classifier head.
Backbone frozen here to match the Stage 1 checkpoint shape; unfrozen
in the next section.


In [ ]:
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

# Freeze backbone (matches the architecture used to produce the Stage 1 checkpoint)
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier head for our 7 classes
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)

# Only the new classifier head is trainable at this point
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

## 11. Class Weights for Imbalanced Loss

Inverse-frequency class weights, passed to the loss function.


In [ ]:
class_counts = Counter(train_dataset.targets)
total_samples = sum(class_counts.values())

class_weights = torch.tensor(
    [total_samples / (NUM_CLASSES * class_counts[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float,
).to(device)

print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)

## 12. Stage 2 — Load Stage 1 Weights & Unfreeze All Layers

Load the Stage 1 best checkpoint, unfreeze every layer, and use a low
learning rate for full-network fine-tuning.


In [ ]:
best_model_path = r"D:\NIH Dataset\Results\Stage1\best_model.pth"
model.load_state_dict(torch.load(STAGE1_MODEL_PATH, map_location=device))

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 13. Training & Validation Loop

Train/validate each epoch, save a checkpoint every epoch, and save the
model weights whenever validation accuracy improves.


In [ ]:
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []
best_val_acc = BEST_VAL_ACC_BASELINE

for epoch in range(NUM_EPOCHS):

    # -------- Train --------
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_acc = 100 * correct / total
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # -------- Validate --------
    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_loader)
    epoch_val_acc = 100 * correct / total
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    # -------- Checkpoint --------
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,
        "best_val_acc": best_val_acc,
    }, CHECKPOINT_PATH)

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Best model saved!")

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_acc:.2f}% "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_acc:.2f}%"
    )

## 14. Evaluate the Best Model on the Test Set

Reload the best checkpoint and run inference on the test set.


In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print("Best model loaded for testing")

In [ ]:
all_labels, all_preds = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)


## 15. Compute & Save Evaluation Metrics

Accuracy, precision, recall, F1-score, saved to `metrics.txt`.


In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="weighted")
recall = recall_score(all_labels, all_preds, average="weighted")
f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

with open(os.path.join(RESULTS_DIR, "metrics.txt"), "w") as f:
    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1-score : {f1:.4f}\n")


## 16. Confusion Matrix

Per-class prediction heatmap, saved as `confusion_matrix.png`.


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - DenseNet121")
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=300, bbox_inches="tight")
plt.show()


## 17. Classification Report

Per-class precision/recall/F1, saved as `classification_report.txt`.


In [ ]:
report = classification_report(all_labels, all_preds, target_names=SELECTED_CLASSES)
print(report)

with open(os.path.join(RESULTS_DIR, "classification_report.txt"), "w") as f:
    f.write(report)


## 18. Training Curves

Loss and accuracy over epochs, saved as `loss_curve.png` and
`accuracy_curve.png`.


In [ ]:
# Loss curve
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.savefig(os.path.join(RESULTS_DIR, "loss_curve.png"), dpi=300, bbox_inches="tight")
plt.show()

# Accuracy curve
plt.figure(figsize=(8, 5))
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.savefig(os.path.join(RESULTS_DIR, "accuracy_curve.png"), dpi=300, bbox_inches="tight")
plt.show()
